# Import

In [2]:
import os
import random
import glob
import re

import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from tqdm import tqdm


### Fix GPU

In [26]:
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

# Fixed RandomSeed & Setting Hyperparameter

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [28]:
LOOKBACK, PREDICT, BATCH_SIZE, EPOCHS = 28, 7, 64, 50
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Data Load

In [29]:
train = pd.read_csv('./train/train.csv')

# Define Model

In [30]:
class MultiOutputLSTM(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=3, output_dim=7):
        super(MultiOutputLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])  # (B, output_dim)

# Train

In [31]:
def train_lstm(train_df):
    trained_models = {}

    for store_menu, group in tqdm(train_df.groupby(['영업장명_메뉴명']), desc ='Training LSTM'):
        store_train = group.sort_values('영업일자').copy()
        if len(store_train) < LOOKBACK + PREDICT:
            continue

        features = ['매출수량']
        scaler = MinMaxScaler()
        store_train[features] = scaler.fit_transform(store_train[features])
        train_vals = store_train[features].values  # shape: (N, 1)

        # 시퀀스 구성
        X_train, y_train = [], []
        for i in range(len(train_vals) - LOOKBACK - PREDICT + 1):
            X_train.append(train_vals[i:i+LOOKBACK])
            y_train.append(train_vals[i+LOOKBACK:i+LOOKBACK+PREDICT, 0])

        X_train = torch.tensor(X_train).float().to(DEVICE)
        y_train = torch.tensor(y_train).float().to(DEVICE)

        model = MultiOutputLSTM(input_dim=1, output_dim=PREDICT).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.MSELoss()

        model.train()
        for epoch in range(EPOCHS):
            idx = torch.randperm(len(X_train))
            for i in range(0, len(X_train), BATCH_SIZE):
                batch_idx = idx[i:i+BATCH_SIZE]
                X_batch, y_batch = X_train[batch_idx], y_train[batch_idx]
                output = model(X_batch)
                loss = criterion(output, y_batch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        trained_models[store_menu] = {
            'model': model.eval(),
            'scaler': scaler,
            'last_sequence': train_vals[-LOOKBACK:]  # (28, 1)
        }

    return trained_models

In [ ]:
# 학습
trained_models = train_lstm(train)

Training LSTM:   0%|          | 0/193 [00:00<?, ?it/s]

# Prediction

In [ ]:
def predict_lstm(test_df, trained_models, test_prefix: str):
    results = []

    for store_menu, store_test in test_df.groupby(['영업장명_메뉴명']):
        key = store_menu
        if key not in trained_models:
            continue

        model = trained_models[key]['model']
        scaler = trained_models[key]['scaler']

        store_test_sorted = store_test.sort_values('영업일자')
        recent_vals = store_test_sorted['매출수량'].values[-LOOKBACK:]
        if len(recent_vals) < LOOKBACK:
            continue

        # 정규화
        recent_vals = scaler.transform(recent_vals.reshape(-1, 1))
        x_input = torch.tensor([recent_vals]).float().to(DEVICE)

        with torch.no_grad():
            pred_scaled = model(x_input).squeeze().cpu().numpy()

        # 역변환
        restored = []
        for i in range(PREDICT):
            dummy = np.zeros((1, 1))
            dummy[0, 0] = pred_scaled[i]
            restored_val = scaler.inverse_transform(dummy)[0, 0]
            restored.append(max(restored_val, 0))

        # 예측일자: TEST_00+1일 ~ TEST_00+7일
        pred_dates = [f"{test_prefix}+{i+1}일" for i in range(PREDICT)]

        for d, val in zip(pred_dates, restored):
            results.append({
                '영업일자': d,
                '영업장명_메뉴명': store_menu,
                '매출수량': val
            })

    return pd.DataFrame(results)


In [16]:
all_preds = []

# 모든 test_*.csv 순회
test_files = sorted(glob.glob('./test/TEST_*.csv'))

for path in test_files:
    test_df = pd.read_csv(path)

    # 파일명에서 접두어 추출 (예: TEST_00)
    filename = os.path.basename(path)
    test_prefix = re.search(r'(TEST_\d+)', filename).group(1)

    pred_df = predict_lstm(test_df, trained_models, test_prefix)
    all_preds.append(pred_df)
    
full_pred_df = pd.concat(all_preds, ignore_index=True)

full_pred_df


/home/wonjun/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/home/wonjun/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/home/wonjun/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/home/wonjun/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/home/wonjun/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
/home/wonjun/.l

,영업일자,영업장명_메뉴명,매출수량
0,TEST_00+1일,"(느티나무 셀프BBQ_1인 수저세트,)",6.719226
1,TEST_00+2일,"(느티나무 셀프BBQ_1인 수저세트,)",3.162589
2,TEST_00+3일,"(느티나무 셀프BBQ_1인 수저세트,)",1.370174
3,TEST_00+4일,"(느티나무 셀프BBQ_1인 수저세트,)",2.720053
4,TEST_00+5일,"(느티나무 셀프BBQ_1인 수저세트,)",5.801484
...,...,...,...
13505,TEST_09+3일,"(화담숲카페_현미뻥스크림,)",20.101147
13506,TEST_09+4일,"(화담숲카페_현미뻥스크림,)",16.664272
13507,TEST_09+5일,"(화담숲카페_현미뻥스크림,)",24.481022
13508,TEST_09+6일,"(화담숲카페_현미뻥스크림,)",27.789471


## Submission (cwj)

In [23]:
sample_submission = pd.read_csv('./sample_submission.csv')

final_df = sample_submission.copy()

# Indexing -> 영업일자를 인덱스로 설정
final_df.set_index('영업일자', inplace=True)


# 3) full_pred_df의 각 행을 돌며 값 채우기
for _, row in full_pred_df.iterrows():
    date = row['영업일자']
    menu = row['영업장명_메뉴명']
    qty  = row['매출수량']
    
    # 행과 열이 모두 존재할 때에만 할당
    if date in final_df.index and menu[0] in final_df.columns:
        final_df.at[date, menu[0]] = qty

# 4) (필요시) '영업일자'를 다시 컬럼으로 돌려놓기
final_df.reset_index(inplace=True)

# 5) 결과 확인
print(final_df.head())

final_df.to_csv('baseline_submission.csv', index=False, encoding='utf-8-sig')

/tmp/ipykernel_2173429/327588077.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6.719225645065308' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.at[date, menu[0]] = qty
/tmp/ipykernel_2173429/327588077.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '12.02039735019207' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.at[date, menu[0]] = qty
/tmp/ipykernel_2173429/327588077.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '7.740222215652466' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.at[date, menu[0]] = qty
/tmp/ipykernel_2173429/327588077.py:17: FutureWarning: Setting an item

         영업일자  느티나무 셀프BBQ_1인 수저세트  느티나무 셀프BBQ_BBQ55(단체)  \
0  TEST_00+1일            6.719226              0.000000   
1  TEST_00+2일            3.162589             12.020397   
2  TEST_00+3일            1.370174             17.498350   
3  TEST_00+4일            2.720053             32.345904   
4  TEST_00+5일            5.801484             61.899665   

   느티나무 셀프BBQ_대여료 30,000원  느티나무 셀프BBQ_대여료 60,000원  느티나무 셀프BBQ_대여료 90,000원  \
0                7.740222                3.500982                0.523461   
1                2.223160                0.545433                0.000000   
2                4.004215                0.926689                0.000000   
3                2.310755                1.696525                0.188686   
4                3.013395                1.222519                0.198248   

   느티나무 셀프BBQ_본삼겹 (단품,실내)  느티나무 셀프BBQ_스프라이트 (단체)  느티나무 셀프BBQ_신라면  \
0                1.541901               2.369392        2.343577   
1                1.223572               4.4644

/tmp/ipykernel_2173429/327588077.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  final_df.reset_index(inplace=True)


# Submission

In [24]:
def convert_to_submission_format(pred_df: pd.DataFrame, sample_submission: pd.DataFrame):
    # (영업일자, 메뉴) → 매출수량 딕셔너리로 변환
    pred_dict = dict(zip(
        zip(pred_df['영업일자'], pred_df['영업장명_메뉴명'][0]),
        pred_df['매출수량']
    ))

    final_df = sample_submission.copy()

    for row_idx in final_df.index:
        date = final_df.loc[row_idx, '영업일자']
        for col in final_df.columns[1:]:  # 메뉴명들
            final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)

    return final_df

In [25]:
sample_submission = pd.read_csv('./sample_submission.csv')
submission = convert_to_submission_format(full_pred_df, sample_submission)
submission.to_csv('baseline22_submission.csv', index=False, encoding='utf-8-sig')

/tmp/ipykernel_2173429/108552172.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '6.719225645065308' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  final_df.loc[row_idx, col] = pred_dict.get((date, col), 0)
